In [19]:
import streamlit as st
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [20]:

st.set_page_config(page_title="FallUp Scraper", layout="centered")

# --- Keyword mapping ---
INDUSTRY_KEYWORDS = {
    "Artificial Intelligence": ["ai", "machine learning", "deep learning", "neural network"],
    "Information Technology": ["cloud", "software", "infrastructure", "devops", "it services"],
    "Human Resources": ["recruitment", "talent", "hr software", "onboarding", "employee"],
    "E-commerce": ["shop", "cart", "checkout", "ecommerce", "store", "buy"],
    "Finance": ["investment", "banking", "fintech", "insurance", "financial"],
    "Healthcare": ["healthcare", "clinic", "hospital", "medical", "pharma"]
}

# --- Helper Functions ---
def normalize_url(domain):
    if not domain.startswith("http"):
        return "http://" + domain
    return domain

def classify_industry(text):
    if not isinstance(text, str):
        return "Unknown"
    text = text.lower()
    for industry, keywords in INDUSTRY_KEYWORDS.items():
        if any(keyword in text for keyword in keywords):
            return industry
    return "Other"

def scrape_website(domain):
    url = normalize_url(domain)
    try:
        response = requests.get(url, timeout=8)
        soup = BeautifulSoup(response.content, "html.parser")
        title = soup.title.string if soup.title else "No title"
        text = soup.get_text()
        industry = classify_industry(text)

        links = [a.get("href") for a in soup.find_all("a", href=True)]
        linkedin = next((l for l in links if "linkedin.com" in l), None)
        twitter = next((l for l in links if "twitter.com" in l), None)
        facebook = next((l for l in links if "facebook.com" in l), None)

        return {
            "Domain": domain,
            "Title": title,
            "Predicted Industry": industry,
            "LinkedIn": linkedin,
            "Twitter": twitter,
            "Facebook": facebook
        }
    except Exception as e:
        return {"Domain": domain, "Error": str(e)}

# --- Streamlit UI ---
st.title("FallUp Domain Scraper")
st.markdown("Enter a domain to extract industry & social info:")

scraped_data = []
domain_input = st.text_input("Enter Domain (e.g., example.com)")

if st.button("Scrape Now") and domain_input:
    with st.spinner("Scraping the website..."):
        result = scrape_website(domain_input)
    scraped_data.append(result)
    st.success("Scraping complete!")
    for key, value in result.items():
        st.markdown(f"**{key}:** {value}")

    # Convert to DataFrame for download
    df_result = pd.DataFrame([result])
    csv = df_result.to_csv(index=False).encode('utf-8')
    st.download_button(
        label="📥 Download Result as CSV",
        data=csv,
        file_name=f"scraped_{domain_input.replace('.', '_')}.csv",
        mime="text/csv"
    )


In [17]:
import pandas as pd
import altair as alt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [18]:
app_code = """
import streamlit as st
import pandas as pd
import altair as alt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

st.set_page_config(page_title="Lead Scoring App", layout="wide")

# Custom CSS for branding and color
custom_css = r\"\"\"
<style>
    html, body, [class*="css"] {
        background-color: #f7f9fc !important;
        color: #333333 !important;
    }
    h1 {
        background-color: #2c3e50 !important;
        color: white !important;
        padding: 1rem !important;
        border-radius: 8px !important;
        text-align: center !important;
    }
    button[kind="primary"] {
        background-color: #3498db !important;
        color: white !important;
        border: none !important;
    }
    button[kind="secondary"] {
        background-color: #2ecc71 !important;
        color: white !important;
    }
    section[data-testid="stSidebar"] {
        background-color: #ecf0f1 !important;
    }
</style>
\"\"\"
st.markdown(custom_css, unsafe_allow_html=True)

st.markdown(r\"\"\"
    <h1 style='text-align: center; color: #2c3e50; font-size: 3.2em; font-weight: bold;'>
        🚀 <span style="color:#3498db">Fall</span><span style="color:#2ecc71">Up</span> Lead Scoring App
    </h1>
\"\"\", unsafe_allow_html=True)
st.markdown("Upload your company data and get intelligent scoring based on employee size, growth, email domain, leadership presence, and more.")

uploaded_file = st.file_uploader("Upload your CSV file", type=["csv"])

def compute_star_rating(row):
    score = 0

    # Employee count score (0–1)
    emp = row.get("number_of_employees", 0)
    if emp >= 5000:
        score += 1
    elif emp >= 1000:
        score += 0.75
    elif emp >= 100:
        score += 0.5
    else:
        score += 0.25

    # Share price growth
    try:
        growth = float(row.get("share_price_current", 0)) - float(row.get("share_price_5y_ago", 0))
        if growth > 200:
            score += 1
        elif growth > 100:
            score += 0.75
        elif growth > 0:
            score += 0.5
        else:
            score += 0.25
    except:
        score += 0.25

    # Email domain quality
    email = str(row.get("contact_email", ""))
    domain = email.split("@")[-1] if "@" in email else ""
    generic_domains = ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com"]
    score += 1 if domain not in generic_domains else 0.25

    # LinkedIn presence
    linkedin = row.get("linkedin_contact", "")
    if isinstance(linkedin, str) and "linkedin.com" in linkedin:
        score += 1

    # CEO/Founder bonus
    bonus = 0
    if isinstance(row.get("ceo_name", ""), str) and row["ceo_name"].strip():
        bonus += 0.5
    if isinstance(row.get("founder_name", ""), str) and row["founder_name"].strip():
        bonus += 0.5

    return round(min(score + bonus, 5), 1)

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
    df["star_rating"] = df.apply(compute_star_rating, axis=1)

    # Add clustering features
    df["has_ceo"] = df["ceo_name"].apply(lambda x: 1 if isinstance(x, str) and x.strip() else 0)
    df["has_email"] = df["contact_email"].apply(lambda x: 1 if isinstance(x, str) and "@" in x else 0)
    df["has_linkedin"] = df["linkedin_contact"].apply(lambda x: 1 if isinstance(x, str) and "linkedin.com" in x else 0)
    df["high_value_domain"] = df["contact_email"].apply(
        lambda x: 1 if isinstance(x, str) and not any(d in x for d in ["gmail.com","yahoo.com","hotmail.com","outlook.com"]) else 0
    )

    features = ["has_ceo","has_email","has_linkedin","high_value_domain"]
    X = df[features].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=4, random_state=42)
    df["cluster"] = kmeans.fit_predict(X_scaled)
    cluster_quality = df.groupby("cluster")[features].mean().sum(axis=1)
    cluster_score_map = cluster_quality.rank(method="dense", ascending=False).astype(int)
    df["cluster_score"] = df["cluster"].map(cluster_score_map)

    # Sidebar filters
    st.sidebar.title("📊 Filters")
    if "industry" in df.columns:
        industries = df["industry"].dropna().unique()
        sel_ind = st.sidebar.multiselect("Industry", industries)
        if sel_ind:
            df = df[df["industry"].isin(sel_ind)]
    if "country" in df.columns:
        countries = df["country"].dropna().unique()
        sel_ct = st.sidebar.multiselect("Country", countries)
        if sel_ct:
            df = df[df["country"].isin(sel_ct)]
    min_s, max_s = st.sidebar.slider("Score Range", 0.0, 5.0, (0.0,5.0), step=0.1)
    df = df[(df["star_rating"]>=min_s)&(df["star_rating"]<=max_s)]

    # Display results
    st.subheader("Top 10 Companies by Score")
    top_df = df.sort_values(by=["star_rating","cluster_score"], ascending=[False,False]).head(10)
    st.dataframe(top_df)

    # Download button
    csv = df.to_csv(index=False).encode("utf-8")
    st.download_button("📥 Download Scored Results", csv, "scored_companies.csv", "text/csv")
"""


In [8]:
with open(r"C:\Users\hp\Documents\fallupapp.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("Saved as fallupapp.py")


Saved as fallupapp.py
